# অলীকবচন — Joggota + BanglaBERT Pipeline
**Self-contained notebook.** No external `.py` files needed.

### Setup Checklist:
1. GPU: **T4 x2** or **P100**
2. Attach fine-tuned BanglaBERT-large weights as a Kaggle Dataset → path: `/kaggle/input/banglabert-finetuned-hallu`
3. Attach mDeBERTa weights as a Kaggle Dataset → update `NLI_MODEL_NAME` below
4. Attach LaBSE weights as a Kaggle Dataset → update `EMBED_MODEL_NAME` below
5. Upload `offline_corpus.json` alongside this notebook (or as a Dataset)
6. Upload `dataset samples.json` and `test set.csv`
7. For Phase 2: toggle **Internet OFF** before running

In [ ]:
!pip install -q accelerate sentence-transformers xgboost


## Part 1: Joggota Engine (Deterministic Rules + Mini-RAG)

In [ ]:
import math
import re
import os
import json
import numpy as np
from collections import Counter
import pandas as pd

# ==========================================
# 0. OFFLINE CORPUS RETRIEVER (Mini-RAG)
# ==========================================

CORPUS_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "offline_corpus.json")

def _bn_words(text):
    """Tokenize Bengali text into words (Bengali unicode + ASCII tokens)."""
    return re.findall(r"[\u0980-\u09FF]+|\w+", str(text).lower())

class CorpusRetriever:
    """
    Lightweight TF-IDF retrieval engine.
    Loads offline_corpus.json once at init. For a given query,
    returns the best-matching paragraph and a similarity score.
    Zero external dependencies beyond numpy.
    """
    def __init__(self, corpus_path=CORPUS_PATH):
        self.paragraphs = []
        self.sources = []
        self.vocab = {}        # word -> index
        self.idf = None        # numpy array
        self.tfidf_matrix = None  # (n_docs, vocab_size)
        
        if not os.path.exists(corpus_path):
            print(f"[CorpusRetriever] WARNING: {corpus_path} not found. Retrieval features disabled.")
            self.enabled = False
            return
        
        with open(corpus_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        self.paragraphs = [d["text"] for d in data]
        self.sources = [d["source"] for d in data]
        
        if len(self.paragraphs) == 0:
            self.enabled = False
            return
        
        self.enabled = True
        self._build_index()
        print(f"[CorpusRetriever] Loaded {len(self.paragraphs)} paragraphs from corpus.")

    def _build_index(self):
        """Build TF-IDF vectors for all paragraphs."""
        # Step 1: Build vocabulary from all documents
        doc_word_counts = []
        doc_freq = Counter()  # how many docs contain each word
        
        for para in self.paragraphs:
            words = _bn_words(para)
            wc = Counter(words)
            doc_word_counts.append(wc)
            for w in set(words):
                doc_freq[w] += 1
        
        # Only keep words that appear in >= 2 docs (filters noise)
        vocab_words = [w for w, freq in doc_freq.items() if freq >= 2]
        self.vocab = {w: i for i, w in enumerate(vocab_words)}
        V = len(self.vocab)
        N = len(self.paragraphs)
        
        # Step 2: Compute IDF
        self.idf = np.zeros(V)
        for w, idx in self.vocab.items():
            self.idf[idx] = math.log((N + 1) / (doc_freq[w] + 1)) + 1  # smoothed IDF
        
        # Step 3: Build TF-IDF matrix
        self.tfidf_matrix = np.zeros((N, V))
        for doc_i, wc in enumerate(doc_word_counts):
            total = sum(wc.values())
            for w, c in wc.items():
                if w in self.vocab:
                    tf = c / total
                    self.tfidf_matrix[doc_i, self.vocab[w]] = tf * self.idf[self.vocab[w]]
        
        # Normalize rows for cosine similarity
        norms = np.linalg.norm(self.tfidf_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1
        self.tfidf_matrix = self.tfidf_matrix / norms

    def query(self, text, top_k=1):
        """
        Returns (best_score, best_paragraph, best_source) for the given text.
        Score is cosine similarity (0 to 1). Higher = better match.
        """
        if not self.enabled:
            return 0.0, "", ""
        
        words = _bn_words(text)
        if not words:
            return 0.0, "", ""
        
        # Build query vector
        wc = Counter(words)
        total = sum(wc.values())
        q_vec = np.zeros(len(self.vocab))
        for w, c in wc.items():
            if w in self.vocab:
                tf = c / total
                q_vec[self.vocab[w]] = tf * self.idf[self.vocab[w]]
        
        q_norm = np.linalg.norm(q_vec)
        if q_norm == 0:
            return 0.0, "", ""
        q_vec = q_vec / q_norm
        
        # Cosine similarity against all docs
        scores = self.tfidf_matrix @ q_vec
        best_idx = np.argmax(scores)
        return float(scores[best_idx]), self.paragraphs[best_idx], self.sources[best_idx]

    def score_grounding(self, prompt, response):
        """
        Composite grounding score: how well does the response match
        corpus evidence retrieved for the prompt?
        Returns a float 0-1. High = response is well-grounded.
        """
        if not self.enabled:
            return 0.0
        
        # Retrieve the best corpus paragraph for this prompt
        prompt_score, best_para, _ = self.query(prompt)
        
        if prompt_score < 0.05:
            # Prompt doesn't match anything in our corpus — can't judge
            return 0.0
        
        # Now check: does the response align with the retrieved evidence?
        resp_score, _, _ = self.query(response)
        
        # Also check overlap between response and the specific retrieved paragraph
        resp_words = set(_bn_words(response))
        para_words = set(_bn_words(best_para))
        if not resp_words:
            return 0.0
        overlap = len(resp_words & para_words) / len(resp_words)
        
        # Blend: retrieval relevance + direct word overlap
        return 0.5 * resp_score + 0.5 * overlap


# Singleton — loaded once when joggota_core is imported
_corpus_retriever = CorpusRetriever()


# ==========================================
# 1. THE FORM ENGINE (Akangkha & Asotti)
# ==========================================

def word_entropy(text):
    """Measures structural randomness. High entropy often correlates with verbose hallucinations."""
    words = str(text).split()
    if len(words) < 2: return 0.0
    counts = Counter(words)
    total = len(words)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())

def char_entropy(text):
    """Character-level entropy."""
    text = str(text)
    if len(text) < 2: return 0.0
    counts = Counter(text)
    total = len(text)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())

def novel_char_ratio(context, response):
    """Detects extrinsic hallucinations by finding characters in response not in context."""
    if pd.isna(context) or str(context).strip() in {"", "[NULL]", "nan", "NaN"}:
        return 0.0 # Only applies to RAG/Context rows
    
    ctx_chars  = set(str(context))
    resp_chars = set(str(response))
    return len(resp_chars - ctx_chars) / max(len(resp_chars), 1)

def response_length_ratio(prompt, response):
    """Hallucinated responses are on average 69% longer in this dataset."""
    p_len = len(str(prompt).strip())
    r_len = len(str(response).strip())
    return r_len / max(p_len, 1)


# ==========================================
# 2. DETERMINISTIC JOGGOTA (Rules & Regex)
# ==========================================

def classify_task(prompt: str) -> str:
    """Classifies the task type to route the validation logic."""
    p = str(prompt).lower()
    if re.search(r'বাগধারা|প্রবাদ|প্রবচন', p): return "idiom"
    if re.search(r'অর্থ|ভাবার্থ|শাব্দিক|সমার্থক|বিপরীত|প্রতিশব্দ', p): return "vocabulary"
    if re.search(r'বানান|শুদ্ধ বানান', p): return "spelling"
    if re.search(r'সম্ভাবনা|যোগ|বিয়োগ|গুণ|ভাগ|সংখ্যা|সমীকরণ|ক্ষেত্রফল|পরিসীমা|লসাগু|গসাগু', p): return "math"
    if re.search(r'অনুবাদ|ইংরেজি|translate', p): return "translation"
    if re.search(r'সমাস|ব্যাকরণ|কারক|বিভক্তি|ধাতু|প্রত্যয়|উপসর্গ', p): return "grammar"
    return "factual"

def deterministic_lexical_joggota(prompt: str, response: str, task_type: str) -> float:
    """
    Applies strict rule-based Joggota for specific task types.
    Returns:
      1.0 (Looks Faithful/Passed rule)
      0.0 (Definite Hallucination)
      -1.0 (Abstain - Rule doesn't apply)
    """
    prompt_clean = str(prompt).strip()
    resp_clean = str(response).strip()
    
    if task_type == "spelling":
        # If it's a spelling task, the response should ideally just be the correct word.
        # If the response is extremely long, it's likely hallucinating a whole explanation.
        if len(resp_clean.split()) > 5:
            return 0.0
            
    if task_type == "math":
        # If math, extract numbers. If there are NO numbers in response, it's likely a hallucinated refusal.
        resp_numbers = re.findall(r'\d+', resp_clean)
        if not resp_numbers:
            return 0.0
            
    if task_type == "idiom":
        # Idioms require figurative meaning, not literal.
        COMMON_IDIOMS = {
            "অকাল কুষ্মাণ্ড": {"literal": ["কুমড়া", "সবজি", "ফল"], "figurative": ["অপদার্থ", "অযোগ্য", "বাজে", "অকাজের"]},
            "আকাশ কুসুম": {"literal": ["আকাশ", "ফুল"], "figurative": ["অসম্ভব", "কল্পনা", "অবাস্তব", "মিথ্যা"]},
            "ঘোড়ার ডিম": {"literal": ["ঘোড়া", "ডিম", "অশ্ব"], "figurative": ["অসম্ভব", "অবাস্তব", "কিছুই না", "অস্তিত্বহীন"]},
            "গাছে কাঁঠাল গোঁফে তেল": {"literal": ["গাছ", "কাঁঠাল", "গোঁফ", "তেল"], "figurative": ["আগে", "প্রস্তুতি", "পাওয়ার", "আশায়"]},
            "চোখে সর্ষে ফুল দেখা": {"literal": ["সর্ষে", "ফুল", "চোখ"], "figurative": ["বিপদ", "দিশেহারা", "ঘোরগ্রস্ত", "অন্ধকার"]},
            "হাতের পাঁচ": {"literal": ["হাত", "পাঁচ", "আঙুল"], "figurative": ["সম্বল", "উপায়", "শেষ", "একমাত্র"]},
            "অন্ধের যষ্টি": {"literal": ["অন্ধ", "লাঠি", "যষ্টি"], "figurative": ["অবলম্বন", "একমাত্র", "ভরসা"]},
            "আদা জল খেয়ে লাগা": {"literal": ["আদা", "জল", "পানি"], "figurative": ["উৎসাহে", "চেষ্টা", "প্রাণপণ", "উঠেপড়ে"]},
            "আঙ্গুল ফুলে কলাগাছ": {"literal": ["আঙ্গুল", "কলাগাছ", "গাছ"], "figurative": ["বড়লোক", "উন্নতি", "ধনী", "হঠাৎ"]}
        }
        
        for idiom, keywords in COMMON_IDIOMS.items():
            if idiom in prompt_clean:
                # Check if the LLM took it literally
                has_literal = any(word in resp_clean for word in keywords["literal"])
                has_figurative = any(word in resp_clean for word in keywords["figurative"])
                
                if has_literal and not has_figurative:
                    return 0.0 # Absolute hallucination
                elif has_figurative:
                    return 1.0 # Passed deterministic check
            
    # For vocabulary, we would ideally hook in the local dictionary check here.
    # For now, we abstain if no strict rule was triggered.
    return -1.0 

def extract_joggota_features(df: pd.DataFrame) -> pd.DataFrame:
    """Applies the Form Engine and Deterministic Joggota to the entire dataframe."""
    df_out = df.copy()
    
    # Clean NaNs
    df_out['context'] = df_out['context'].fillna('[NULL]')
    df_out['prompt_bn'] = df_out['prompt_bn'].astype(str)
    df_out['response_bn'] = df_out['response_bn'].astype(str)
    
    # 1. Task Classification
    df_out['task_type'] = df_out['prompt_bn'].apply(classify_task)
    
    # 2. Form Engine Features
    df_out['word_entropy'] = df_out['response_bn'].apply(word_entropy)
    df_out['char_entropy'] = df_out['response_bn'].apply(char_entropy)
    df_out['novel_char_ratio'] = df_out.apply(lambda row: novel_char_ratio(row['context'], row['response_bn']), axis=1)
    df_out['length_ratio'] = df_out.apply(lambda row: response_length_ratio(row['prompt_bn'], row['response_bn']), axis=1)
    df_out['resp_len'] = df_out['response_bn'].str.len()
    
    # 3. Deterministic Joggota
    df_out['deterministic_joggota'] = df_out.apply(
        lambda row: deterministic_lexical_joggota(row['prompt_bn'], row['response_bn'], row['task_type']), axis=1
    )
    
    # 4. Corpus Retrieval Grounding (Mini-RAG)
    print("Computing corpus grounding scores...")
    df_out['corpus_match_score'] = df_out.apply(
        lambda row: _corpus_retriever.score_grounding(row['prompt_bn'], row['response_bn']), axis=1
    )
    
    return df_out



## Part 2: Train BanglaBERT (Auto-saves to /kaggle/working/banglabert_finetuned/)

In [ ]:
# Auto-training cell: fine-tunes BanglaBERT-large on 299 labeled samples + synthetic pairs
import os, gc, json, random, re
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "csebuetnlp/banglabert_large"
BATCH_SIZE, EPOCHS, LR, MAX_LEN, WARMUP = 8, 3, 1e-5, 256, 0.1
CKPT_DIR = "/kaggle/working/banglabert_finetuned"

# ---- Load & combine datasets ----
def _lp(path):
    with open(path, "r", encoding="utf-8") as f: data = json.load(f)
    pre, resp, lbl = [], [], []
    for r in data:
        ctx = r.get("context", "")
        if pd.isna(ctx) or str(ctx).strip().lower() in ("[null]", "null", "none", "nan", ""):
            premise = str(r["prompt_bn"])
        else: premise = str(r["prompt_bn"]) + " " + str(ctx).strip()
        pre.append(premise); resp.append(str(r["response_bn"])); lbl.append(int(r["label"]))
    return pre, resp, lbl
aug_premises, aug_responses, aug_labels = _lp("dataset samples.json")
# Search for BenHalluEval in /kaggle/input/ (upload as a separate Kaggle dataset named 'benhallueval-training')
BE_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "benhallueval_training.json":
            BE_PATH = os.path.join(root, f); break
if BE_PATH is None and os.path.exists("benhallueval_training.json"): BE_PATH = "benhallueval_training.json"
if BE_PATH:
    bp, br, bl = _lp(BE_PATH)
    aug_premises.extend(bp); aug_responses.extend(br); aug_labels.extend(bl)
    print(f"  + BenHalluEval from {BE_PATH}: {len(bp)} pairs ({sum(bl)} faithful, {len(bl)-sum(bl)} hallucinated)")
else:
    print("  ⚠️ benhallueval_training.json not found — training on competition data only")
# Shuffle combined data
combined = list(zip(aug_premises, aug_responses, aug_labels)); random.shuffle(combined)
aug_premises, aug_responses, aug_labels = [list(x) for x in zip(*combined)] if combined else ([],[],[])
print(f"Total training: {len(aug_labels)} pairs ({sum(aug_labels)} faithful, {len(aug_labels)-sum(aug_labels)} hallucinated)")

# ---- Dataset ----
class TrainPairDS(Dataset):
    def __init__(self, prems, resps, ys, tok, mx=MAX_LEN):
        self.p=prems; self.h=resps; self.y=ys; self.t=tok; self.m=mx
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        e = self.t(self.p[i], self.h[i], truncation=True, max_length=self.m, padding="max_length", return_tensors="pt")
        return {"input_ids": e["input_ids"].squeeze(0), "attention_mask": e["attention_mask"].squeeze(0), "labels": torch.tensor(self.y[i], dtype=torch.long)}

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=1.0):
        super().__init__(); self.g = gamma; self.register_buffer("w", torch.tensor([alpha, 1.0]))
    def forward(self, lg, y):
        ce = F.cross_entropy(lg, y, weight=self.w.to(lg.device), reduction="none"); pt = torch.exp(-ce); return ((1-pt)**self.g*ce).mean()

# ---- Train ----
from sklearn.model_selection import train_test_split
tp, vp, tr, vr, ty, vy = train_test_split(aug_premises, aug_responses, aug_labels, test_size=0.2, random_state=42, stratify=aug_labels)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True).float().to(DEVICE)
tld = DataLoader(TrainPairDS(tp, tr, ty, tok), batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
vld = DataLoader(TrainPairDS(vp, vr, vy, tok), batch_size=BATCH_SIZE*2, shuffle=False, pin_memory=True)
crit = FocalLoss(2.0, 1.0); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
tot = len(tld)*EPOCHS; sch = get_cosine_schedule_with_warmup(opt, int(tot*WARMUP), tot)
scaler = torch.amp.GradScaler("cuda"); best_f1, best_state = 0.0, None

@torch.no_grad()
def evalfn(m, ld):
    m.eval(); ap, al = [], []
    for b in ld:
        with torch.amp.autocast("cuda", dtype=torch.float16): lg = m(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE)).logits.float()
        pr = torch.softmax(lg, -1)[:,1].cpu().numpy(); ap.extend((pr>=0.5).astype(int)); al.extend(b["labels"].numpy())
    return f1_score(al, ap)

print(f"Training {EPOCHS} epochs...")
for ep in range(EPOCHS):
    model.train(); tl=0.0
    for b in tld:
        opt.zero_grad()
        with torch.amp.autocast("cuda", dtype=torch.float16): lg = model(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE)).logits; lo = crit(lg, b["labels"].to(DEVICE))
        scaler.scale(lo).backward(); scaler.step(opt); scaler.update(); sch.step(); tl += lo.item()
    vf = evalfn(model, vld)
    print(f"  Epoch {ep+1}: loss={tl/len(tld):.4f} | val F1={vf:.4f}")
    if vf > best_f1: best_f1 = vf; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None: model.load_state_dict(best_state)
os.makedirs(CKPT_DIR, exist_ok=True)
model.save_pretrained(CKPT_DIR); tok.save_pretrained(CKPT_DIR)
print(f"\n✅ Saved fine-tuned model to {CKPT_DIR}/ (val F1={best_f1:.4f})")
del model, tok, tld, vld; gc.collect(); torch.cuda.empty_cache()
print("Training complete. Ready for inference.")


## Part 3: Submission Pipeline (NLI + BanglaBERT + XGBoost)

In [ ]:
"""
Unified Joggota Submission Pipeline
Merges tithy-4.ipynb (NLI + XGBoost) with joggota_core.py (Form Engine + Deterministic Rules)
+ Fine-tuned BanglaBERT-large Classifier for All Rows

VRAM strategy:
  Phase 1 (NLI + Embeddings) → GPU, then explicit .to("cpu") + empty_cache()
  Phase 2 (BanglaBERT-large) → GPU, batch inference on ALL rows (context + no-context)
  Fallback: If BanglaBERT fails, use Phase 1 sim_premise_response as proxy signal
"""
import os
import gc
import json
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import xgboost as xgb
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader

# Import our custom Joggota rules

# ----------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------
TRAIN_PATH = "dataset samples.json"
TEST_PATH = "test set.csv"
SAMPLE_SUB_PATH = "sample submission.csv"
SUBMISSION_OUT = "submission.csv"

EMBED_MODEL_NAME = "sentence-transformers/LaBSE"
NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# On Kaggle, change this to your offline dataset path, e.g. "/kaggle/input/tigerllm-9b-it"
TIGERLLM_MODEL_NAME = "md-nishat-008/TigerLLM-9B-it"

RANDOM_STATE = 42
N_FOLDS = 5

# ----------------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    df["context"] = df["context"].replace("[NULL]", np.nan)
    df["context"] = df["context"].replace("", np.nan)
    df["has_context"] = df["context"].notna().astype(int)
    return df

def load_test_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "context" not in df.columns:
        df["context"] = np.nan
    df["context"] = df["context"].replace("[NULL]", np.nan)
    df["context"] = df["context"].replace("", np.nan)
    df["has_context"] = df["context"].notna().astype(int)
    return df

# ----------------------------------------------------------------------
# 2. MODELS (Embeddings, NLI, LLM Judge, Fallback)
# ----------------------------------------------------------------------
class Embedder:
    def __init__(self, model_name=EMBED_MODEL_NAME, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_name, device=self.device)

    def encode(self, texts):
        clean = []
        for t in texts:
            if t is None or (isinstance(t, float) and np.isnan(t)):
                clean.append("।")
            else:
                t = str(t)
                clean.append(t if t.strip() else "।")
        return self.model.encode(clean, batch_size=32, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

    def cosine(self, a_emb, b_emb):
        return np.sum(a_emb * b_emb, axis=1)

class NLIScorer:
    def __init__(self, model_name=NLI_MODEL_NAME, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()

    @torch.no_grad()
    def score_batch(self, premises, hypotheses, batch_size=16):
        premises = ["" if p is None or (isinstance(p, float) and np.isnan(p)) else str(p) for p in premises]
        hypotheses = ["" if h is None or (isinstance(h, float) and np.isnan(h)) else str(h) for h in hypotheses]
        premises = [p if p.strip() else "।" for p in premises]
        hypotheses = [h if h.strip() else "।" for h in hypotheses]

        all_probs = []
        for i in range(0, len(premises), batch_size):
            p_batch = premises[i:i + batch_size]
            h_batch = hypotheses[i:i + batch_size]
            inputs = self.tokenizer(p_batch, h_batch, truncation=True, padding=True, max_length=256, return_tensors="pt").to(self.device)
            logits = self.model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
        return np.vstack(all_probs)

def _force_gpu_cleanup():
    """Aggressively clear GPU memory."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    # Small allocation to defragment
    try:
        _dummy = torch.zeros(1, device="cuda")
        del _dummy
    except:
        pass

class PairDataset(Dataset):
    """Minimal (premise, response) dataset for BanglaBERT cross-encoder inference."""
    def __init__(self, premises, responses, tokenizer, max_len=384):
        self.premises = list(premises)
        self.responses = list(responses)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, i):
        enc = self.tokenizer(
            str(self.premises[i]), str(self.responses[i]),
            truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


class BanglaBERTClassifier:
    """
    Fine-tuned BanglaBERT-large cross-encoder for faithfulness scoring.
    Loads from a local checkpoint if available, otherwise from HuggingFace.
    350M params — fits comfortably on T4 alongside other models.
    """
    BANGLA_MODEL = "csebuetnlp/banglabert_large"

    def __init__(self, checkpoint_path=None, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        model_path = checkpoint_path or self.BANGLA_MODEL
        print(f"Loading BanglaBERT classifier from {model_path} on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_path, num_labels=2, ignore_mismatched_sizes=True
        ).half().eval().to(self.device)

    @torch.no_grad()
    def score_all_rows(self, df, batch_size=32):
        """
        Score ALL rows (context + no-context) in batches.
        Uses context-or-prompt as premise, response as hypothesis.
        Returns a numpy array of P(faithful) probabilities (0–1).
        """
        out = np.zeros(len(df))
        # Build premise: context if available, else prompt
        premises = []
        for _, r in df.iterrows():
            ctx = r.get("context", "")
            if pd.isna(ctx) or str(ctx).strip().lower() in ("[null]", "null", "none", "nan", ""):
                premises.append(str(r["prompt_bn"]))
            else:
                premises.append(str(r["prompt_bn"]) + " " + str(ctx).strip())
        responses = df["response_bn"].astype(str).tolist()

        ds = PairDataset(premises, responses, self.tokenizer, max_len=384)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False, pin_memory=True)

        all_probs = []
        for batch in loader:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                logits = self.model(
                    input_ids=batch["input_ids"].to(self.device),
                    attention_mask=batch["attention_mask"].to(self.device)
                ).logits.float()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.append(probs)

        probs = np.concatenate(all_probs)
        out[:] = probs
        return out


# ----------------------------------------------------------------------
# 3. LEXICAL HELPERS (From tithy-4)
# ----------------------------------------------------------------------
def bn_tokenize(text):
    if not isinstance(text, str):
        if text is None or (isinstance(text, float) and np.isnan(text)): return []
        text = str(text)
    return re.findall(r"[\u0980-\u09FF]+|\S+", text)

def token_overlap_ratio(a, b):
    ta, tb = set(bn_tokenize(a)), set(bn_tokenize(b))
    if not ta or not tb: return 0.0
    return len(ta & tb) / len(ta | tb)

# ----------------------------------------------------------------------
# 4. UNIFIED FEATURE PIPELINE
# ----------------------------------------------------------------------
def build_base_features(df, embedder, nli_scorer):
    print("Extracting Form & Deterministic Joggota features...")
    df = extract_joggota_features(df)
    
    prompt_col = df["prompt_bn"]
    response_col = df["response_bn"]
    context_col = df["context"]
    
    print("Running NLI...")
    premise_ctx = context_col.fillna(prompt_col)
    probs_ctx = nli_scorer.score_batch(premise_ctx.tolist(), response_col.tolist())
    df["nli_ctx_entail"] = probs_ctx[:, 0]
    df["nli_ctx_contra"] = probs_ctx[:, 2]

    print("Running Embeddings...")
    ctx_or_prompt_emb = embedder.encode(premise_ctx.tolist())
    resp_emb = embedder.encode(response_col.tolist())
    df["sim_premise_response"] = embedder.cosine(ctx_or_prompt_emb, resp_emb)
    
    df["token_overlap_ctx_resp"] = [token_overlap_ratio(p, r) for p, r in zip(premise_ctx, response_col)]

    feature_cols = [
        "nli_ctx_entail", "nli_ctx_contra", "sim_premise_response", "token_overlap_ctx_resp", "has_context",
        "word_entropy", "char_entropy", "novel_char_ratio", "length_ratio", "deterministic_joggota",
        "corpus_match_score"
    ]
    df[feature_cols] = df[feature_cols].fillna(0)
    
    return df, feature_cols

# ----------------------------------------------------------------------
# 5. XGBOOST TRAINING & INFERENCE
# ----------------------------------------------------------------------
# ==========================================
# 6. RUN LOGGER — tracks per-run statistics
# ==========================================
RUN_LOG_PATH = "run_log.csv"

def _log_dataset_stats(df, name):
    """Log basic dataset statistics."""
    n = len(df)
    n_ctx = df["has_context"].sum()
    n_noctx = n - n_ctx
    print(f"\n{'='*60}")
    print(f"📊 {name} — {n} rows ({n_ctx} with context, {n_noctx} without)")
    if "label" in df.columns:
        n_faithful = df["label"].sum()
        n_hallu = n - n_faithful
        print(f"   Labels: {n_faithful} faithful ({n_faithful/n*100:.1f}%) | {n_hallu} hallucinated ({n_hallu/n*100:.1f}%)")
    return n, n_ctx, n_noctx

def _log_task_distribution(df):
    """Print task type breakdown."""
    if "task_type" not in df.columns:
        return
    print("\n📂 Task Distribution:")
    for task, count in df["task_type"].value_counts().items():
        pct = count / len(df) * 100
        print(f"   {task:20s}: {count:4d} ({pct:.1f}%)")

def _log_feature_summary(df, feature_cols):
    """Print mean/std for each numerical feature."""
    print("\n📈 Feature Summary (mean ± std across train+test):")
    for col in feature_cols:
        if col in df.columns:
            vals = df[col].dropna()
            mean = vals.mean()
            std = vals.std()
            print(f"   {col:30s}: {mean:.4f} ± {std:.4f}")

def _log_final_distribution(preds, test_df):
    """Print what the pipeline predicted."""
    total = len(preds)
    faithful = preds.sum()
    hallu = total - faithful
    print(f"\n🎯 Final Predictions: {faithful} faithful ({faithful/total*100:.1f}%) | {hallu} hallucinated ({hallu/total*100:.1f}%)")
    print(f"   (out of {total} test rows)")

def _save_run_log(train_df, test_df, feature_cols, cv_results=None, used_fallback=False):
    """Append a row to run_log.csv for git-pull insights."""
    if "label" not in train_df.columns:
        return
    row = {
        "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "train_rows": len(train_df),
        "train_faithful": int(train_df["label"].sum()),
        "train_hallucinated": int((1 - train_df["label"]).sum()),
        "train_context": int(train_df["has_context"].sum()),
        "train_no_context": int((1 - train_df["has_context"]).sum()),
        "test_rows": len(test_df),
        "test_context": int(test_df["has_context"].sum()),
        "test_no_context": int((1 - test_df["has_context"]).sum()),
        "used_fallback": int(used_fallback),
    }
    for col in feature_cols:
        if col in train_df.columns:
            row[f"train_{col}_mean"] = float(train_df[col].mean())
            row[f"train_{col}_std"] = float(train_df[col].std())
    if cv_results is not None:
        row["cv_f1_mean"] = float(np.mean(cv_results))
        row["cv_f1_std"] = float(np.std(cv_results))
    
    log_df = pd.DataFrame([row])
    try:
        existing = pd.read_csv(RUN_LOG_PATH)
        log_df = pd.concat([existing, log_df], ignore_index=True)
    except FileNotFoundError:
        pass
    log_df.to_csv(RUN_LOG_PATH, index=False)
    print(f"\n📝 Run logged to {RUN_LOG_PATH}")

def _cross_validate(X, y, n_folds=5):
    """Run stratified k-fold CV and return per-fold F1 scores."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        model = xgb.XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE
        )
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        f1 = f1_score(y_val, preds)
        scores.append(f1)
        print(f"   Fold {fold+1}: F1 = {f1:.4f}")
    mean_f1 = np.mean(scores)
    std_f1 = np.std(scores)
    print(f"   {'='*30}")
    print(f"   CV F1: {mean_f1:.4f} ± {std_f1:.4f}")
    return scores

def train_and_predict():
    print("Loading data...")
    train_df = load_json(TRAIN_PATH)
    test_df = load_test_csv(TEST_PATH)
    
    _log_dataset_stats(train_df, "TRAIN SET")
    _log_dataset_stats(test_df, "TEST SET")
    
    # --- PHASE 1: NLI & Embeddings (GPU) ---
    print("\nLoading NLI & Embedding models...")
    embedder = Embedder()
    nli_scorer = NLIScorer()
    
    print("\n--- PHASE 1: Base Features ---")
    print("--- Processing Train Set ---")
    train_df, feature_cols = build_base_features(train_df, embedder, nli_scorer)
    
    print("\n--- Processing Test Set ---")
    test_df, _ = build_base_features(test_df, embedder, nli_scorer)
    
    _log_task_distribution(train_df)
    _log_task_distribution(test_df)
    
    # Aggressive GPU cleanup
    print("\nCleaning up Phase 1 GPU memory...")
    embedder.model.to("cpu")
    nli_scorer.model.to("cpu")
    del embedder, nli_scorer
    _force_gpu_cleanup()

    # --- PHASE 2: BanglaBERT Classifier (All Rows) ---
    used_fallback = False
    print("\n--- PHASE 2: BanglaBERT Faithfulness Scoring ---")
    
    try:
        # Require a fine-tuned checkpoint — base model has a random head
        checkpoint = None
        for candidate in (
            "/kaggle/working/banglabert_finetuned",          # Auto-trained by notebook Part 2
            "/kaggle/input/banglabert-finetuned-hallu",      # Pre-attached Kaggle dataset
            "banglabert_finetuned",                          # Local training output
            "banglabert_checkpoint.pt",                      # Legacy checkpoint
        ):
            if os.path.isdir(candidate) or (os.path.isfile(candidate) and candidate.endswith(".pt")):
                checkpoint = candidate
                break

        if checkpoint is None:
            raise FileNotFoundError(
                "No fine-tuned BanglaBERT checkpoint found. "
                "Skipping to sim_premise_response fallback — "
                "base banglabert_large has a random classification head "
                "and would inject noise into XGBoost."
            )

        banglabert = BanglaBERTClassifier(checkpoint_path=checkpoint)
        train_df["banglabert_faithful_prob"] = banglabert.score_all_rows(train_df, batch_size=32)
        test_df["banglabert_faithful_prob"] = banglabert.score_all_rows(test_df, batch_size=32)
        feature_cols.append("banglabert_faithful_prob")

        train_bb = train_df["banglabert_faithful_prob"]
        test_bb = test_df["banglabert_faithful_prob"]
        n_train_scored = (train_bb > 0).sum()
        n_test_scored = (test_bb > 0).sum()
        print(f"\n🧠 BanglaBERT Score Summary:")
        print(f"   Train: mean={train_bb.mean():.3f} | rows scored={n_train_scored}/{len(train_df)}")
        print(f"   Test:  mean={test_bb.mean():.3f} | rows scored={n_test_scored}/{len(test_df)}")

        del banglabert
        _force_gpu_cleanup()
    except Exception as e:
        print(f"⚠️ BanglaBERT failed: {e}")
        print(f"   → Falling back to Phase 1 sim_premise_response as proxy signal...")
        used_fallback = True

        # Use already-computed LaBSE prompt-response similarity from Phase 1
        # (no additional model load needed)
        train_df["banglabert_faithful_prob"] = train_df["sim_premise_response"]
        test_df["banglabert_faithful_prob"] = test_df["sim_premise_response"]
        feature_cols.append("banglabert_faithful_prob")

        train_bb = train_df["banglabert_faithful_prob"]
        test_bb = test_df["banglabert_faithful_prob"]
        print(f"\n🔤 sim_premise_response fallback summary:")
        print(f"   Train: mean={train_bb.mean():.3f}")
        print(f"   Test:  mean={test_bb.mean():.3f}")

    # ---- Feature summary ----
    combined = pd.concat([train_df, test_df], ignore_index=True)
    _log_feature_summary(combined, feature_cols)

    # --- PHASE 3: XGBoost ---
    print("\n--- PHASE 3: XGBoost Training & Cross-Validation ---")
    X_train = train_df[feature_cols].values
    y_train = train_df["label"].values
    X_test = test_df[feature_cols].values
    
    print("\n📐 5-Fold Cross-Validation:")
    cv_scores = _cross_validate(X_train, y_train, n_folds=N_FOLDS)
    
    print("\n🏋️  Training final model on full training set...")
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    
    importance = model.feature_importances_
    print("\n🔍 Feature Importance:")
    for col, imp in sorted(zip(feature_cols, importance), key=lambda x: -x[1]):
        print(f"   {col:30s}: {imp:.4f}")
    
    print("\nPredicting on test set...")
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)
    
    _log_final_distribution(preds, test_df)
    
    submission = pd.DataFrame({"id": test_df["id"].values, "label": preds})
    submission.to_csv(SUBMISSION_OUT, index=False)
    print(f"\n✅ Saved submission → {SUBMISSION_OUT}")
    
    _save_run_log(train_df, test_df, feature_cols, cv_scores, used_fallback)
    
    if used_fallback:
        print(f"\n📌 Note: Used sim_premise_response fallback (BanglaBERT couldn't load)")
    print(f"✅ Run complete.")

if __name__ == "__main__":
    train_and_predict()


## Part 4: Run Inference

In [ ]:
train_and_predict()
